In [11]:
import wandb
from wandb.integration.keras import WandbCallback
from dotenv import load_dotenv
import utils
import tensorflow as tf
from keras import layers
import models_base, models_top

In [12]:
import os
from dotenv import load_dotenv

# 환경변수 설정
os.environ['WANDB_AGENT_DISABLE_FLAPPING'] = 'true'

load_dotenv()
WAND_B_API_KEY = os.getenv("WAND_B_API_KEY")

wandb.login(key=WAND_B_API_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/dataliteracy/.netrc


True

In [3]:
# Sweep 설정
sweep_config = {
    "name": "ev2l_top3_sweep",
    "metric": {"name": "val_loss", "goal": "minimize"},
    "method": "random",
    "parameters": {
        "learning_rate": {"min": 0.001, "max": 0.01},
        "epochs": {"values": [5, 10, 15]},
        "batch_size": {"values": [32, 64, 128]},
    }
}

In [4]:
# Sweep ID 생성
sweep_id = wandb.sweep(sweep_config, entity="ljhlovecgy0212-aiffel", project="jellyfish")
print(f"Sweep ID: {sweep_id}")

Create sweep with ID: fb6z3m6s
Sweep URL: https://wandb.ai/ljhlovecgy0212-aiffel/jellyfish/sweeps/fb6z3m6s
Sweep ID: fb6z3m6s


In [5]:
# 데이터 증강 설정
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
])

2025-01-23 08:47:11.112705: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2025-01-23 08:47:11.112752: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
2025-01-23 08:47:11.112755: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.00 GB
2025-01-23 08:47:11.112780: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-01-23 08:47:11.112797: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [6]:
# 모델 정의
def ev2l_top3():
    inputs = tf.keras.Input(shape=(224, 224, 3))
    augmented_inputs = data_augmentation(inputs)

    x = models_base.EV2L((224, 224, 3))(augmented_inputs, training=False)
    x = models_top.top3(x)
    outputs = layers.Dense(6, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

In [9]:
# Sweep에서 실행할 학습 함수
def train():
    with wandb.init(settings=wandb.Settings(init_timeout=1600)) as run:
        config = wandb.config

        # 데이터셋 로드
        train_dataset, val_dataset, test_dataset = utils.load_datasets("data/data_no_aug", batch_size=config.batch_size)

        # 모델 생성
        model = ev2l_top3()
        model.compile(
            optimizer=tf.keras.optimizers.RMSprop(learning_rate=config.learning_rate),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        # 모델 학습
        model.fit(
            train_dataset,
            validation_data=val_dataset,
            epochs=config.epochs,
            callbacks=[
                utils.callback_earlystop(10),  # Early stopping
                WandbCallback(log_weights=True, save_model=False, save_graph=True)  # W&B 콜백
            ]
        )

        # 모델 평가
        test_loss, test_accuracy = model.evaluate(test_dataset, verbose=2)
        wandb.log({"test_loss": test_loss, "test_accuracy": test_accuracy})

In [10]:
# Sweep 실행
wandb.agent(sweep_id, function=train, count=10)

wandb.finish()

wandb: Agent Starting Run: 8bjqrhig with config:
wandb: 	batch_size: 128
wandb: 	epochs: 10
wandb: 	learning_rate: 0.008094036888111304
wandb: ERROR Run 8bjqrhig errored:
wandb: ERROR Traceback (most recent call last):
wandb: ERROR   File "/Users/dataliteracy/Library/Python/3.11/lib/python/site-packages/wandb/agents/pyagent.py", line 306, in _run_job
wandb: ERROR     self._function()
wandb: ERROR   File "/var/folders/t8/x1j50zj52n5064gxgs7q3drc0000gn/T/ipykernel_6473/3209188222.py", line 3, in train
wandb: ERROR     with wandb.init(settings=wandb.Settings(init_timeout=1600)) as run:
wandb: ERROR          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
wandb: ERROR   File "/Users/dataliteracy/Library/Python/3.11/lib/python/site-packages/wandb/sdk/wandb_init.py", line 1458, in init
wandb: ERROR     wandb._sentry.reraise(e)
wandb: ERROR   File "/Users/dataliteracy/Library/Python/3.11/lib/python/site-packages/wandb/analytics/sentry.py", line 156, in reraise
wandb: ERROR     raise ex